# PINN demo (benchmark B2)

Trains and evaluates the physics-informed model on a coarse grid and a short L-BFGS budget, so the whole notebook runs in about a minute on a CPU. This is a smoke test of the pipeline, not the reported result -- the head error here is orders of magnitude larger than the reported 0.0147 m, because the model barely trains.

For the reported configuration and the sweeps this trainer also runs, see [`experiments/README.md`](../experiments/README.md) and the main [README's Training section](../README.md#training-a-model).

Requires `GW_DATA` set before launching Jupyter, or the data placed under `data/` at the repository root -- see the main README's [Data availability](../README.md#data-availability).

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path.cwd().parent  # this notebook lives in notebooks/
DATA = Path(os.environ.get("GW_DATA", REPO / "data"))
PINN_DIR = REPO / "src" / "pinn" / "b2"
EVALUATOR = REPO / "src" / "diagnostics" / "evaluate_b2.py"

field_pattern = str(DATA / "benchmarks" / "b2" / "t*.txt")
anchor_pattern = str(DATA / "benchmarks" / "b2" / "sdata" / "t*.txt")

print("repository:", REPO)
print("data root: ", DATA)

## Stage 1 -- short initial window

In [ ]:
common = [
    "--chrono", "--constraint", "HARD",
    "--spatial_strategy", "UNIFORM", "--nx", "24", "--ny", "24",
    "--temporal_strategy", "LHS", "--nt", "6", "--tau", "1", "--sigma", "30",
    "--anchor_pattern", anchor_pattern, "--field_pattern", field_pattern,
    "--epochs_Adam", "1", "--epochs_LBFGS", "20", "--lbfgs_max_iter", "3",
    "--alpha_fixed", "0.05",
]

subprocess.run(["python", "train.py", "--stage", "1", *common],
               cwd=PINN_DIR, check=True)

## Stage 2 -- full horizon, resumes from stage 1

`--spatial_strategy LR`, the setting every reported run uses, needs a precomputed collocation cloud (`--filename`); `UNIFORM` samples on the fly and needs nothing beyond the flags below.

In [ ]:
subprocess.run(
    ["python", "train.py", "--stage", "2", *common,
     "--temporal_strategy_prev", "LHS", "--nt_prev", "6"],
    cwd=PINN_DIR, check=True,
)

## Test on the locked days (26-30)

In [ ]:
subprocess.run(
    ["python", "test.py", "--stage", "2", "--chrono", "--constraint", "HARD",
     "--spatial_strategy", "UNIFORM", "--nx", "24", "--ny", "24", "--sigma", "30",
     "--temporal_strategy", "LHS", "--nt", "6",
     "--temporal_strategy_prev", "LHS", "--nt_prev", "6",
     "--anchor_pattern", anchor_pattern, "--field_pattern", field_pattern],
    cwd=PINN_DIR, check=True,
)

## Conservation diagnostics on the test predictions

In [ ]:
pred_dir = sorted((PINN_DIR / "outputs_chrono_b2").glob(
    "Stage2_*/test_predictions"))[-1]

subprocess.run(
    ["python", str(EVALUATOR),
     "--prediction-dir", str(pred_dir),
     "--output-dir", str(REPO / "out" / "pinn_demo"),
     "--model-name", "PINN B2 demo",
     "--prediction-mode", "direct", "--no-plots"],
    check=True,
)

The reported PINN reaches R² > 0.9999 and a residual ratio of 0.51 at the full `nt = 50`, thousands of L-BFGS epochs, and the precomputed collocation cloud -- see the main README's results table.